<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>Dataset과 DataLoader(Data Pipeline)</strong>에 대해 학습합니다.</br></br>
>PyTorch의 데이터 파이프라인을 구성하고 배치 단위 학습을 학습해봅시다.

</br>

# Dataset
> PyTorch에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">데이터를 추상화</mark>하는 기본 클래스입니다.

> 학습 데이터를 한꺼번에 메모리에 올리면 ImageNet처럼 수백 GB에 달하는 데이터셋에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">메모리 부족(OOM)</mark> 오류가 발생합니다.</br></br>
> Dataset 추상화는 `__getitem__`으로 필요한 샘플만 그때그때 불러오는 지연 로딩(lazy loading)을 가능하게 합니다. 또한 샘플을 하나씩 모델에 넣으면 GPU 병렬 연산을 활용하지 못하는데, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">DataLoader</mark>는 여러 샘플을 배치 텐서로 묶어 GPU 처리 효율을 극대화합니다.</br></br>
> `shuffle=True`로 매 에폭마다 순서를 무작위로 섞어 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">일반화 성능을 향상</mark>시키고, `num_workers > 0` 설정으로 데이터 로딩을 별도 프로세스에서 미리 수행하여 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">파이프라인 병렬성</mark>을 실현할 수 있습니다.

## TensorDataset
> 텐서를 직접 감싸는 가장 간단한 Dataset입니다.

In [2]:
import torch
from torch.utils.data import TensorDataset
# TODO 1: X에 [[1,2],[3,4],[5,6],[7,8]]을 실수형 텐서로, y에 [0,1,0,1]을 정수형 텐서로 생성한 뒤,
# 텐서 데이터셋으로 묶어 길이와 첫 번째 샘플을 출력해봅시다.
# X는 실수형(float32), y는 분류 정답이므로 정수형(long)으로 만듭니다.
X_data = torch.tensor([[1, 2], [3, 4], [5, 6], [7, 8]], dtype=torch.float32)
y_data = torch.tensor([0, 1, 0, 1], dtype=torch.long)

# TensorDataset으로 X와 y를 한 쌍으로 묶어줍니다.
tensor_dataset = TensorDataset(X_data, y_data)

print("--- TensorDataset 생성 ---")
print(f"데이터셋 총 길이: {len(tensor_dataset)}")
print(f"첫 번째 샘플 (X, y): {tensor_dataset[0]}")

--- TensorDataset 생성 ---
데이터셋 총 길이: 4
첫 번째 샘플 (X, y): (tensor([1., 2.]), tensor(0))


</br>

## 커스텀 Dataset
> `__len__`과 `__getitem__`을 구현하여 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">자유로운 데이터 로딩</mark>이 가능합니다.

In [4]:
# TODO 2: 커스텀 데이터셋 클래스를 정의하고, 초기화에서 X를 실수형, y를 정수형 텐서로 변환하여 저장하고, 
# 데이터 개수 반환과 인덱스로 샘플을 가져오는 메서드를 구현해봅시다.
from torch.utils.data import Dataset

# 반드시 Dataset을 상속받고 __init__, __len__, __getitem__ 3대장을 구현해야 합니다.
class MyDataset(Dataset):
    def __init__(self, X, y):
        # 파이썬 리스트나 넘파이 배열이 들어와도 텐서로 변환하여 저장하도록 구현
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        # 데이터의 총 개수를 반환
        return len(self.X)

    def __getitem__(self, idx):
        # 인덱스(idx)를 입력받아 해당하는 (데이터, 라벨) 한 쌍을 반환
        return self.X[idx], self.y[idx]

# 파이썬 기본 리스트를 넘겨서 데이터셋 생성해보기
custom_dataset = MyDataset([[1, 2], [3, 4], [5, 6], [7, 8]], [0, 1, 0, 1])

print("---  Custom Dataset 작동 확인 ---")
print(f"커스텀 데이터셋 첫 번째 샘플: {custom_dataset[0]}")

---  Custom Dataset 작동 확인 ---
커스텀 데이터셋 첫 번째 샘플: (tensor([1., 2.]), tensor(0))


</br>

# DataLoader
> Dataset을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">배치 단위로 순회</mark>할 수 있게 하는 반복자(Iterator)입니다.

In [5]:
# TODO 3: 데이터 로더를 import하고, dataset을 batch_size=2, shuffle=True로 감싼 뒤, 반복문으로 각 배치의 shape과 라벨을 출력해봅시다.
from torch.utils.data import DataLoader

# batch_size=2: 전체 4개의 데이터를 2개씩 잘라서 줍니다.
# shuffle=True: 학습할 때 순서를 외우지 못하도록 데이터를 마구 섞습니다.
dataloader = DataLoader(custom_dataset, batch_size=2, shuffle=True)

print("--- DataLoader 배치 순회 ---")
for batch_idx, (batch_X, batch_y) in enumerate(dataloader):
    print(f"[Batch {batch_idx + 1}]")
    print(f"X shape: {batch_X.shape} | X 데이터:\n{batch_X}")
    print(f"y shape: {batch_y.shape} | y 라벨: {batch_y}\n")

--- DataLoader 배치 순회 ---
[Batch 1]
X shape: torch.Size([2, 2]) | X 데이터:
tensor([[5., 6.],
        [7., 8.]])
y shape: torch.Size([2]) | y 라벨: tensor([0, 1])

[Batch 2]
X shape: torch.Size([2, 2]) | X 데이터:
tensor([[3., 4.],
        [1., 2.]])
y shape: torch.Size([2]) | y 라벨: tensor([1, 0])



</br>

## DataLoader 주요 파라미터

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th style="text-align:center">설명</th>
      <th style="text-align:center">기본값</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>batch_size</code></td><td style="text-align:center">배치 크기</td><td style="text-align:center"><code>1</code></td></tr>
    <tr><td style="text-align:center"><code>shuffle</code></td><td style="text-align:center">에폭마다 순서 섞기</td><td style="text-align:center"><code>False</code></td></tr>
    <tr><td style="text-align:center"><code>num_workers</code></td><td style="text-align:center">병렬 데이터 로딩 프로세스 수</td><td style="text-align:center"><code>0</code></td></tr>
    <tr><td style="text-align:center"><code>drop_last</code></td><td style="text-align:center">마지막 불완전 배치 버리기</td><td style="text-align:center"><code>False</code></td></tr>
    <tr><td style="text-align:center"><code>pin_memory</code></td><td style="text-align:center">GPU 전송 최적화</td><td style="text-align:center"><code>False</code></td></tr>
  </tbody>
</table>

💡shuffle 설정
> 학습용: `shuffle=True` (매 에폭 순서 변경으로 일반화 향상)</br>
> 검증/테스트용: `shuffle=False` (재현성 보장)

</br>

## 학습 루프에서의 활용

In [6]:
# TODO 4: 간단한 모델을 만들고, 학습 루프에서 DataLoader를 활용해봅시다.
# 간단한 선형 모델 (입력 2차원 → 출력 2차원/클래스)
import torch.nn as nn
import torch.optim as optim

model = nn.Linear(2, 2) 

# 분류 문제용 손실 함수(CrossEntropy)와 최적화 도구(SGD)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 5 # 전체 데이터를 5번 반복 학습

print("--- 미니 학습 루프 시작 ---")
for epoch in range(epochs):
    epoch_loss = 0.0
    
    # DataLoader를 순회하며 미니 배치 단위로 학습!
    for batch_X, batch_y in dataloader:
        # 1. 기울기 초기화
        optimizer.zero_grad()
        
        # 2. 순전파 (예측)
        predictions = model(batch_X)
        
        # 3. 손실 계산
        loss = criterion(predictions, batch_y)
        
        # 4. 역전파 (기울기 계산)
        loss.backward()
        
        # 5. 가중치 업데이트
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}] | 평균 Loss: {epoch_loss / len(dataloader):.4f}")

--- 미니 학습 루프 시작 ---
Epoch [1/5] | 평균 Loss: 0.8835
Epoch [2/5] | 평균 Loss: 0.9672
Epoch [3/5] | 평균 Loss: 0.8533
Epoch [4/5] | 평균 Loss: 0.9405
Epoch [5/5] | 평균 Loss: 0.9495


💡batch_size와 메모리
> 배치 크기가 클수록 GPU 메모리를 많이 사용합니다.</br>
> `RuntimeError: CUDA out of memory`가 발생하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">배치 크기를 줄이세요</mark>.